In [1]:
#%% 
# LIBRARIES
# ======================================================
import os
import pickle
import itertools

# =====================================================
# DATA MANIPULATION
# =====================================================

import pandas as pd
import numpy as np

# =====================================================
# VISUALIZATION
# =====================================================

import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================
# SCIKIT-LEARN - PREPROCESSING
# =====================================================

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    StandardScaler,
    OrdinalEncoder
)

# =====================================================
# SCIKIT-LEARN - FEATURE ENGINEERING
# =====================================================

from sklearn.decomposition import PCA

from sklearn.feature_selection import (
    mutual_info_classif
)

# =====================================================
# SCIKIT-LEARN - FEATURE IMPORTANCE
# =====================================================

from sklearn.inspection import (
    permutation_importance
)

# =====================================================
# SCIKIT-LEARN - SUPERVISED LEARNING
# =====================================================

# FINAL MODEL
from sklearn.ensemble import (
    RandomForestClassifier
)

# =====================================================
# SCIKIT-LEARN - ENSEMBLE LEARNING
# =====================================================

from sklearn.ensemble import (
    GradientBoostingClassifier,
    AdaBoostClassifier
)

# =====================================================
# SCIKIT-LEARN - CLUSTERING
# =====================================================

from sklearn.cluster import (

    KMeans,
    DBSCAN
)

# =====================================================
# SCIKIT-LEARN - MODEL VALIDATION
# =====================================================

from sklearn.model_selection import (

    cross_val_score,
    StratifiedKFold,
    GridSearchCV
)

# =====================================================
# SCIKIT-LEARN - METRICS
# =====================================================

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,

    classification_report,
    confusion_matrix,

    roc_auc_score,
    roc_curve,

    silhouette_score,
    balanced_accuracy_score,
    davies_bouldin_score
)

# =====================================================
# STATISTICAL TESTS
# =====================================================

from scipy.stats import (

    f_oneway,
    kruskal,
    ttest_ind
)

# =====================================================
# DEEP LEARNING - TENSORFLOW / KERAS
# =====================================================
import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

# =====================================================
# UTILITIES
# =====================================================

from itertools import (
    combinations_with_replacement
)

In [2]:
#%% =====================================================
# DATA LOADER
# =====================================================

class FlightDelayDataLoader:

    def __init__(
        self,
        filename,
        test_size=0.2,
        random_state=42
    ):

        self.filename = filename
        self.test_size = test_size
        self.random_state = random_state

        self.features_train = None
        self.features_test = None

        self.target_train = None
        self.target_test = None

        self.features_train_original = None
        self.features_test_original = None

        self.data_loader()

    # =====================================================
    # LOAD DATA
    # =====================================================

    def data_loader(self):

        print("\n==============================")
        print("LOADING DATASET")
        print("==============================")

        df = pd.read_csv(
            self.filename,
            low_memory=False
        )

        print(f"Original dataset shape: {df.shape}")

        # ------------------------------------------------
        # REMOVE CANCELLED FLIGHTS
        # ------------------------------------------------

        if "CANCELLED" in df.columns:

            df = df[
                df["CANCELLED"] == 0
            ]

            print(
                f"After removing cancelled flights: "
                f"{df.shape}"
            )

        # ------------------------------------------------
        # CREATE TARGET
        # ------------------------------------------------

        df["FLIGHT_DELAYED"] = (

            df["ARR_DELAY"] > 15

        ).astype(int)

        # ------------------------------------------------
        # CORRELATION CHECK
        # ------------------------------------------------

        print("\n==============================")
        print("TOP CORRELATIONS WITH TARGET")
        print("==============================")

        numeric_df = df.select_dtypes(
            include=np.number
        )

        correlations = (

            numeric_df.corr()["FLIGHT_DELAYED"]

            .abs()

            .sort_values(
                ascending=False
            )
        )

        print(
            correlations.head(20)
        )

        # ------------------------------------------------
        # REMOVE LEAKAGE
        # ------------------------------------------------

        leakage_columns = [

            # target leakage
            "ARR_DELAY",
            "AIRLINE_CODE",
            "AIRLINE_DOT",

            "ORIGIN_CITY",
            "DEST_CITY",

            # future information
            "DEP_DELAY",
            "DEP_TIME",
            "ARR_TIME",

            "WHEELS_OFF",
            "WHEELS_ON",

            "TAXI_OUT",
            "TAXI_IN",

            "AIR_TIME",
            "ELAPSED_TIME",

            # delay reasons
            "DELAY_DUE_CARRIER",
            "DELAY_DUE_WEATHER",
            "DELAY_DUE_NAS",
            "DELAY_DUE_SECURITY",
            "DELAY_DUE_LATE_AIRCRAFT",

            # outcome variables
            "CANCELLED",
            "CANCELLATION_CODE",
            "DIVERTED"
        ]

        existing_columns = [

            col

            for col in leakage_columns

            if col in df.columns
        ]

        df.drop(
            columns=existing_columns,
            inplace=True
        )

        print(
            f"\nRemoved {len(existing_columns)} leakage columns."
        )

        # ------------------------------------------------
        # CLEAN DATA
        # ------------------------------------------------

        df.replace(
            [np.inf, -np.inf],
            np.nan,
            inplace=True
        )

        df.fillna(
            0,
            inplace=True
        )

        # ------------------------------------------------
        # FEATURES / TARGET
        # ------------------------------------------------

        X = df.drop(
            columns=["FLIGHT_DELAYED"]
        )

        y = df["FLIGHT_DELAYED"]

        # ------------------------------------------------
        # TARGET DISTRIBUTION
        # ------------------------------------------------

        print("\n==============================")
        print("TARGET DISTRIBUTION")
        print("==============================")

        print("\nFull Dataset:")

        print(
            y.value_counts()
        )

        print(
            y.value_counts(
                normalize=True
            )
        )

        # ------------------------------------------------
        # FEATURE LIST
        # ------------------------------------------------

        print("\n==============================")
        print("FEATURES USED")
        print("==============================")

        print(
            f"Number of features: "
            f"{len(X.columns)}"
        )

        for col in sorted(X.columns):

            print(col)

        # ------------------------------------------------
        # TRAIN TEST SPLIT
        # ------------------------------------------------

        X_train, X_test, y_train, y_test = (

            train_test_split(

                X,
                y,

                test_size=self.test_size,

                random_state=self.random_state,

                stratify=y
            )
        )

        # ------------------------------------------------
        # DISTRIBUTION AFTER SPLIT
        # ------------------------------------------------

        print("\n==============================")
        print("TRAIN DISTRIBUTION")
        print("==============================")

        print(
            y_train.value_counts()
        )

        print(
            y_train.value_counts(
                normalize=True
            )
        )

        print("\n==============================")
        print("TEST DISTRIBUTION")
        print("==============================")

        print(
            y_test.value_counts()
        )

        print(
            y_test.value_counts(
                normalize=True
            )
        )

        # ------------------------------------------------
        # SAVE ORIGINAL DATA
        # ------------------------------------------------

        self.features_train_original = (
            X_train.copy()
        )

        self.features_test_original = (
            X_test.copy()
        )

        # ------------------------------------------------
        # SAVE WORKING DATA
        # ------------------------------------------------

        self.features_train = (
            X_train.copy()
        )

        self.features_test = (
            X_test.copy()
        )

        self.target_train = (
            y_train.copy()
        )

        self.target_test = (
            y_test.copy()
        )

        # ------------------------------------------------
        # FINAL INFO
        # ------------------------------------------------

        print("\n==============================")
        print("TRAIN / TEST SPLIT COMPLETED")
        print("==============================")

        print(
            f"Training samples: "
            f"{X_train.shape}"
        )

        print(
            f"Testing samples: "
            f"{X_test.shape}"
        )

        print(
            f"Number of features: "
            f"{X_train.shape[1]}"
        )

In [3]:
class FlightDelayDataManipulator(FlightDelayDataLoader):

    def __init__(
        self,
        filename,
        test_size=0.2,
        random_state=42
    ):

        super().__init__(
            filename,
            test_size,
            random_state
        )

        self.create_time_features()
        self.create_route_features()
        self.create_airline_features()
        self.create_distance_features()

    # =====================================================
    # AIRLINE NAMES
    # =====================================================
    def create_time_features(self):

        for df in [

            self.features_train,
            self.features_test

        ]:

            if "CRS_DEP_TIME" in df.columns:

                dep = (
                    df["CRS_DEP_TIME"]
                    .astype(str)
                    .str.zfill(4)
                )

                hour = dep.str[:2].astype(int)

                df["departure_hour"] = hour

                df["is_peak_hour"] = (
                    hour.isin(
                        [7,8,9,17,18,19]
                    )
                ).astype(int)

                df["is_night_flight"] = (
                    hour <= 5
                ).astype(int)

        print("Time features created.")
   
    def create_route_features(self):

        for df in [

            self.features_train,
            self.features_test

        ]:

            if (
                "ORIGIN" in df.columns and
                "DEST" in df.columns
            ):

                df["route"] = (

                    df["ORIGIN"].astype(str)

                    + "_"

                    + df["DEST"].astype(str)

                )

        print("Route features created.")
    def create_airline_features(self):

        airline_map = {

            "AA": "American Airlines",
            "DL": "Delta Airlines",
            "UA": "United Airlines",
            "WN": "Southwest",
            "B6": "JetBlue",
            "AS": "Alaska Airlines",
            "NK": "Spirit",
            "F9": "Frontier"
        }

        for df in [

            self.features_train,
            self.features_test

        ]:

            if "AIRLINE" in df.columns:

                df["airline_name"] = (

                    df["AIRLINE"]

                    .map(airline_map)

                    .fillna("Other")
                )

        print("Airline features created.")
    def create_distance_features(self):

        for df in [

            self.features_train,
            self.features_test

        ]:

            if "DISTANCE" in df.columns:

                df["is_long_flight"] = (

                    df["DISTANCE"] > 1500

                ).astype(int)

        print("Distance features created.")

In [4]:
#%% =====================================================
# DATA CLEANING
# =====================================================

class FlightDelayDataCleaning:

    def __init__(self, data_loader):

        self.data_loader = data_loader


    # =====================================================
    # REMOVE DUPLICATES
    # =====================================================
    def remove_duplicates(self):

        df = self.data_loader.features_train
        y = self.data_loader.target_train

        before = df.shape[0]

        # remover duplicados mantendo alinhamento com target
        df = df.loc[~df.duplicated()]
        y = y.loc[df.index]

        self.data_loader.features_train = df
        self.data_loader.target_train = y

        after = df.shape[0]

        print("Removed duplicates:", before - after)


    # =====================================================
    # HANDLE MISSING VALUES
    # =====================================================
    def handle_missing_values(self):

        train_df = self.data_loader.features_train
        test_df = self.data_loader.features_test

        # separar tipos
        numeric_cols = train_df.select_dtypes(include=["int64", "float64"]).columns
        categorical_cols = train_df.select_dtypes(include=["object", "string", "category"]).columns

        # ----------------------------------------
        # NUMERIC
        # ⚠️ usar estatísticas do treino (evita leakage)
        # ----------------------------------------
        medians = train_df[numeric_cols].median()

        train_df[numeric_cols] = train_df[numeric_cols].fillna(medians)
        test_df[numeric_cols] = test_df[numeric_cols].fillna(medians)

        # ----------------------------------------
        # CATEGORICAL
        # ----------------------------------------
        train_df[categorical_cols] = train_df[categorical_cols].fillna("Unknown")
        test_df[categorical_cols] = test_df[categorical_cols].fillna("Unknown")

        # ----------------------------------------
        # FEATURES GERADAS (muito importante)
        # ----------------------------------------
        # Algumas features novas podem ficar com NaN após .map()

        generated_cols = [
            "airline_delay_rate",
            "origin_delay_rate",
            "month_delay_rate",
            "route_delay_rate",
            "airline_month_delay",
            "dow_delay_rate",
            "airport_airline_delay"
        ]

        for col in generated_cols:
            if col in train_df.columns:
                train_df[col] = train_df[col].fillna(train_df[col].mean())
                test_df[col] = test_df[col].fillna(train_df[col].mean())

        # ----------------------------------------
        # FEATURES DE CONTAGEM
        # ----------------------------------------
        count_cols = [
            "airport_total_traffic",
            "origin_congestion",
            "hour_traffic"
        ]

        for col in count_cols:
            if col in train_df.columns:
                train_df[col] = train_df[col].fillna(0)
                test_df[col] = test_df[col].fillna(0)

        print("Missing values handled correctly")

In [5]:
class FlightDelayPreprocessing:

    def __init__(self, data_loader):

        self.data_loader = data_loader

        self.data_loader.features_train_original = (
            self.data_loader.features_train.copy()
        )

        self.data_loader.features_test_original = (
            self.data_loader.features_test.copy()
        )

        self.preprocess()

    def preprocess(self):

        train_df = self.data_loader.features_train.copy()
        test_df = self.data_loader.features_test.copy()

        # -------------------------------------
        # COLUMN TYPES
        # -------------------------------------

        numeric_cols = train_df.select_dtypes(
            include=np.number
        ).columns.tolist()

        categorical_cols = train_df.select_dtypes(
            include=["object", "string", "category"]
        ).columns.tolist()

        # -------------------------------------
        # REMOVE EDA COLUMNS
        # -------------------------------------

        cols_to_exclude = [

            "airline_name",
            "origin_city_clean",
            "dest_city_clean"

        ]

        categorical_cols = [

            c
            for c in categorical_cols
            if c not in cols_to_exclude
        ]

        # -------------------------------------
        # ORDINAL ENCODING
        # -------------------------------------

        encoder = OrdinalEncoder(

            handle_unknown="use_encoded_value",
            unknown_value=-1

        )

        train_cat = pd.DataFrame(

            encoder.fit_transform(
                train_df[categorical_cols]
            ),

            columns=categorical_cols,
            index=train_df.index
        )

        test_cat = pd.DataFrame(

            encoder.transform(
                test_df[categorical_cols]
            ),

            columns=categorical_cols,
            index=test_df.index
        )

        # -------------------------------------
        # SCALING NUMERIC
        # -------------------------------------

        scaler = StandardScaler()

        train_num = pd.DataFrame(

            scaler.fit_transform(
                train_df[numeric_cols]
            ),

            columns=numeric_cols,
            index=train_df.index
        )

        test_num = pd.DataFrame(

            scaler.transform(
                test_df[numeric_cols]
            ),

            columns=numeric_cols,
            index=test_df.index
        )

        # -------------------------------------
        # FINAL DATASET
        # -------------------------------------

        self.data_loader.features_train = pd.concat(
            [train_num, train_cat],
            axis=1
        )

        self.data_loader.features_test = pd.concat(
            [test_num, test_cat],
            axis=1
        )

        print("\nPreprocessing completed")

        print(
            "Final train shape:",
            self.data_loader.features_train.shape
        )

        print(
            "Final test shape:",
            self.data_loader.features_test.shape
        )

In [6]:
#%% =====================================================
# EXPLORATORY DATA ANALYSIS
# =====================================================

class FlightDelayEDA:

    def __init__(self, data_loader, output_folder="../output/plots"):

        self.data_loader = data_loader
        self.output_folder = output_folder

        os.makedirs(self.output_folder, exist_ok=True)


    def run(self):

        self.plot_correlation_heatmap()
        self.plot_arrived_flights_by_airline()
        self.plot_delays_by_airline()
        self.plot_delay_percentage_by_airline()
        self.plot_delay_categories_global()
        self.plot_delay_categories_by_airline()
        self.plot_flights_by_airport()
        self.plot_arrival_delay_percentage_by_airport()
        self.plot_delay_percentage_by_airport()
        self.plot_airport_traffic()


    # ------------------------------------------------
    # CORRELATION HEATMAP
    # ------------------------------------------------
    def plot_correlation_heatmap(self):

        print("\nGenerating correlation heatmap...")

        df = self.data_loader.features_train.copy()

        # apenas numéricas
        df = df.select_dtypes(include=np.number)

        # adicionar target
        df["FLIGHT_DELAYED"] = (
            self.data_loader.target_train.values
        )

        corr_matrix = df.corr(
            numeric_only=True
        )

        plt.figure(figsize=(12, 10))

        sns.heatmap(
            corr_matrix,
            annot=True,
            cmap="coolwarm",
            fmt=".2f",
            center=0
        )

        plt.title(
            "Correlation Heatmap"
        )

        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/correlation_heatmap.png"
        )

        plt.close()

        print("Correlation heatmap saved.")


    # ------------------------------------------------
    # VOOS POR COMPANHIA
    # ------------------------------------------------
    def plot_arrived_flights_by_airline(self):

        df = self.data_loader.features_train_original.copy()

        if "airline_name" not in df.columns:
            print("airline_name not available")
            return

        counts = df["airline_name"].value_counts()

        plt.figure(figsize=(12,6))
        sns.barplot(x=counts.index, y=counts.values)

        plt.title("Number of Flights per Airline")
        plt.xlabel("Airline")
        plt.ylabel("Number of Flights")

        plt.xticks(rotation=45)
        plt.tight_layout()

        plt.savefig(f"{self.output_folder}/arrived_flights_by_airline.png")
        plt.close()

        print("Flights per airline saved.")


    # ------------------------------------------------
    # ATRASOS POR COMPANHIA
    # ------------------------------------------------
    def plot_delays_by_airline(self):

        df = self.data_loader.features_train_original.copy()
        df["DELAYED"] = self.data_loader.target_train.values

        if "airline_name" not in df.columns:
            print("airline_name not available")
            return

        delays = df[df["DELAYED"] == 1]["airline_name"].value_counts()

        if delays.empty:
            print("No delays found")
            return

        plt.figure(figsize=(12,6))
        sns.barplot(x=delays.index, y=delays.values)

        plt.title("Number of Delayed Flights per Airline")
        plt.xlabel("Airline")
        plt.ylabel("Number of Delays")

        plt.xticks(rotation=45)
        plt.tight_layout()

        plt.savefig(f"{self.output_folder}/delays_per_airline.png")
        plt.close()

        print("Delays per airline saved.")


    # ------------------------------------------------
    # % ATRASOS POR COMPANHIA
    # ------------------------------------------------
    def plot_delay_percentage_by_airline(self):

        df = self.data_loader.features_train_original.copy()
        df["DELAYED"] = self.data_loader.target_train.values

        if "airline_name" not in df.columns:
            return

        delay_rate = df.groupby("airline_name")["DELAYED"].mean() * 100

        plt.figure(figsize=(12,6))
        sns.barplot(x=delay_rate.index, y=delay_rate.values)

        plt.title("Delay Percentage per Airline")
        plt.xlabel("Airline")
        plt.ylabel("Delay (%)")

        plt.xticks(rotation=45)
        plt.tight_layout()

        plt.savefig(f"{self.output_folder}/delay_percentage_per_airline.png")
        plt.close()

        print("Delay percentage per airline saved.")


    def plot_delay_categories_global(self):

        df = self.data_loader.features_train_original.copy()

        if "ARR_DELAY" not in df.columns:
            print("ARR_DELAY not available → skipping")
            return

        def categorize(delay):
            if delay < 15:
                return "On-time"
            elif delay <= 30:
                return "Short delay"
            else:
                return "Long delay"

        df["delay_category"] = df["ARR_DELAY"].apply(categorize)

        counts = df["delay_category"].value_counts()

        plt.figure(figsize=(8,6))
        sns.barplot(x=counts.index, y=counts.values)

        plt.title("Global Delay Categories")
        plt.xlabel("Category")
        plt.ylabel("Number of Flights")

        plt.tight_layout()
        plt.savefig(f"{self.output_folder}/delay_categories_global.png")
        plt.close()

        print("Global delay categories saved.")


    # ------------------------------------------------
    # CATEGORIAS POR COMPANHIA
    # ------------------------------------------------
    def plot_delay_categories_by_airline(self):

        df = self.data_loader.features_train_original.copy()

        if "ARR_DELAY" not in df.columns:
            print("ARR_DELAY not available → skipping")
            return

        def categorize(delay):
            if delay < 15:
                return "On-time"
            elif delay <= 30:
                return "Short delay"
            else:
                return "Long delay"

        df["delay_category"] = df["ARR_DELAY"].apply(categorize)

        if "airline_name" not in df.columns:
            return

        grouped = df.groupby(
            ["airline_name", "delay_category"]
        ).size().reset_index(name="count")

        plt.figure(figsize=(14,7))
        sns.barplot(
            data=grouped,
            x="airline_name",
            y="count",
            hue="delay_category"
        )

        plt.title("Delay Categories by Airline")
        plt.xlabel("Airline")
        plt.ylabel("Number of Flights")

        plt.xticks(rotation=45)
        plt.tight_layout()

        plt.savefig(f"{self.output_folder}/delay_categories_by_airline.png")
        plt.close()

        print("Delay categories by airline saved.")


    # ------------------------------------------------
    # VOOS POR AEROPORTO
    # ------------------------------------------------
    def plot_flights_by_airport(self):

        df = self.data_loader.features_train_original.copy()

        if "ORIGIN" in df.columns:

            departures = df["ORIGIN"].value_counts().head(15)

            plt.figure(figsize=(12,6))
            sns.barplot(x=departures.index, y=departures.values)

            plt.title("Top Airports by Departures")
            plt.xticks(rotation=45)

            plt.tight_layout()
            plt.savefig(f"{self.output_folder}/departures_by_airport.png")
            plt.close()

        if "DEST" in df.columns:

            arrivals = df["DEST"].value_counts().head(15)

            plt.figure(figsize=(12,6))
            sns.barplot(x=arrivals.index, y=arrivals.values)

            plt.title("Top Airports by Arrivals")
            plt.xticks(rotation=45)

            plt.tight_layout()
            plt.savefig(f"{self.output_folder}/arrivals_by_airport.png")
            plt.close()

        print("Flights by airport saved.")


    # ------------------------------------------------
    # PIE CHART - CHEGADAS
    # ------------------------------------------------
    def plot_arrival_delay_percentage_by_airport(self, top_n=5):

        df = self.data_loader.features_train_original.copy()

        if "ARR_DELAY" not in df.columns:
            print("ARR_DELAY not available → skipping")
            return

        
        df["DELAYED"] = (df["ARR_DELAY"] > 15).astype(int)

        top_airports = df["DEST"].value_counts().head(top_n).index

        for airport in top_airports:

            airport_df = df[df["DEST"] == airport]

            delayed = airport_df["DELAYED"].sum()
            ontime = len(airport_df) - delayed

            if delayed == 0 or ontime == 0:
                continue

            plt.figure(figsize=(6,6))
            plt.pie([ontime, delayed], labels=["On-time", "Delayed"], autopct="%1.1f%%")

            plt.title(f"Arrival Delays - {airport}")
            plt.savefig(f"{self.output_folder}/arrival_delay_{airport}.png")
            plt.close()

        print("Arrival pie charts saved.")


    # ------------------------------------------------
    # PIE CHART - PARTIDAS
    # ------------------------------------------------
    def plot_delay_percentage_by_airport(self, top_n=5):

        df = self.data_loader.features_train_original.copy()

        if "ARR_DELAY" not in df.columns:
            print("ARR_DELAY not available → skipping")
            return
        
        df["DELAYED"] = (df["ARR_DELAY"] > 15).astype(int)

        top_airports = df["ORIGIN"].value_counts().head(top_n).index

        for airport in top_airports:

            airport_df = df[df["ORIGIN"] == airport]

            delayed = airport_df["DELAYED"].sum()
            ontime = len(airport_df) - delayed

            if delayed == 0 or ontime == 0:
                continue

            plt.figure(figsize=(6,6))
            plt.pie([ontime, delayed], labels=["On-time", "Delayed"], autopct="%1.1f%%")

            plt.title(f"Departure Delays - {airport}")
            plt.savefig(f"{self.output_folder}/departure_delay_{airport}.png")
            plt.close()

        print("Departure pie charts saved.")


    # ------------------------------------------------
    # TRAFEGO AEROPORTO
    # ------------------------------------------------
    def plot_airport_traffic(self):

        df = self.data_loader.features_train_original.copy()

        if "ORIGIN" not in df.columns:
            return

        traffic = df["ORIGIN"].value_counts().head(15)

        plt.figure(figsize=(12,6))
        sns.barplot(x=traffic.index, y=traffic.values)

        plt.title("Top Airports by Traffic")
        plt.xticks(rotation=45)

        plt.tight_layout()
        plt.savefig(f"{self.output_folder}/airport_traffic.png")
        plt.close()

        print("Airport traffic saved.")

In [7]:
#%% =====================================================
# FEATURE ANALYSIS
# =====================================================

# =====================================================
# FEATURE ANALYSIS
# =====================================================

class FlightDelayFeatureAnalysis:

    def __init__(
        self,
        data_loader,
        output_folder="outputs/plots"
    ):

        self.data_loader = data_loader
        self.output_folder = output_folder

        os.makedirs(
            self.output_folder,
            exist_ok=True
        )


    # =====================================================
    # PCA ANALYSIS
    # =====================================================

    def perform_pca(
        self,
        explained_variance_threshold=0.80,
        plot_pca=True,
        add_pca=True
    ):

        train_df = (
            self.data_loader.features_train.copy()
        )

        test_df = (
            self.data_loader.features_test.copy()
        )

        # =====================================================
        # KEEP ONLY NUMERIC FEATURES
        # =====================================================

        numeric_train = train_df.select_dtypes(
            include=np.number
        )

        numeric_test = test_df.select_dtypes(
            include=np.number
        )

        if numeric_train.empty:

            raise ValueError(
                "No numeric data available for PCA."
            )

        # =====================================================
        # CLEAN DATA
        # =====================================================

        numeric_train = (
            numeric_train
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
            .astype(np.float32)
        )

        numeric_test = (
            numeric_test
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
            .astype(np.float32)
        )

        # =====================================================
        # SCALE DATA
        # =====================================================

        scaler = StandardScaler()

        scaled_train = scaler.fit_transform(
            numeric_train
        )

        scaled_test = scaler.transform(
            numeric_test
        )

        # =====================================================
        # INITIAL PCA
        # =====================================================

        pca_temp = PCA()

        pca_temp.fit(scaled_train)

        cumulative_variance = np.cumsum(
            pca_temp.explained_variance_ratio_
        )

        n_components = np.argmax(
            cumulative_variance >= explained_variance_threshold
        ) + 1

        print(
            f"Selected {n_components} PCA components"
        )

        # =====================================================
        # FINAL PCA
        # =====================================================

        pca = PCA(
            n_components=n_components
        )

        transformed_train = pca.fit_transform(
            scaled_train
        )

        transformed_test = pca.transform(
            scaled_test
        )

        # =====================================================
        # ADD PCA FEATURES
        # =====================================================

        if add_pca:

            for i in range(n_components):

                train_df[f"PCA_{i+1}"] = (
                    transformed_train[:, i]
                )

                test_df[f"PCA_{i+1}"] = (
                    transformed_test[:, i]
                )

            self.data_loader.features_train = (
                train_df
            )

            self.data_loader.features_test = (
                test_df
            )

        # =====================================================
        # PCA PLOT
        # =====================================================

        if plot_pca:

            plt.figure(figsize=(8,5))

            plt.plot(
                cumulative_variance,
                marker="o"
            )

            plt.axhline(
                y=explained_variance_threshold,
                color="r",
                linestyle="--"
            )

            plt.title(
                "Cumulative Explained Variance (PCA)"
            )

            plt.xlabel(
                "Number of Components"
            )

            plt.ylabel(
                "Variance Explained"
            )

            plt.tight_layout()

            plt.savefig(
                f"{self.output_folder}/pca_variance.png"
            )

            plt.close()

            print("PCA variance plot saved.")


    # =====================================================
    # FEATURE IMPORTANCE
    # =====================================================

    def relevant_feature_identification(
        self,
        num_features=10
    ):

        X = (
            self.data_loader.features_train
            .select_dtypes(include=np.number)
            .copy()
        )

        y = self.data_loader.target_train.copy()

        # =====================================================
        # CLEAN DATA
        # =====================================================

        X = (
            X
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
            .astype(np.float32)
        )

        # =====================================================
        # MEMORY CONTROL
        # =====================================================

        if X.shape[1] > 100:

            print(
                "Too many features detected."
            )

            print(
                "Reducing feature space..."
            )

            variances = X.var()

            selected_columns = (
                variances
                .sort_values(ascending=False)
                .head(100)
                .index
            )

            X = X[selected_columns]

        # =====================================================
        # MUTUAL INFORMATION
        # =====================================================

        mi_scores = mutual_info_classif(
            X,
            y,
            random_state=42
        )

        mi_df = pd.DataFrame({

            "feature": X.columns,

            "importance": mi_scores

        })

        mi_df = mi_df.sort_values(
            by="importance",
            ascending=False
        )

        top_features = mi_df.head(
            num_features
        )

        print("\nTop relevant features:\n")

        print(top_features)

        # =====================================================
        # PLOT
        # =====================================================

        plt.figure(figsize=(10,6))

        sns.barplot(

            data=top_features,

            x="importance",

            y="feature"

        )

        plt.title(
            "Top Feature Importance (Mutual Information)"
        )

        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/feature_importance_MI.png"
        )

        plt.close()

        print(
            "Feature importance plot saved."
        )

        return top_features[
            "feature"
        ].tolist()

#%% =====================================================
# FEATURE GENERATION
# =====================================================

class FlightDelayFeatureAnalysisAndGenerator(
    FlightDelayFeatureAnalysis
):

    def __init__(self, data_loader):

        super().__init__(data_loader)


    # =====================================================
    # MAIN
    # =====================================================

    def generate_features(self):

        self.add_time_features()

        self.add_route_features()

        self.add_interaction_features()

        print("Feature generation completed.")


    # =====================================================
    # TIME FEATURES
    # =====================================================

    def add_time_features(self):

        df_train = self.data_loader.features_train

        df_test = self.data_loader.features_test

        for df in [df_train, df_test]:

            # -----------------------------------------
            # MORNING / AFTERNOON / NIGHT
            # -----------------------------------------

            if "departure_hour" in df.columns:

                df["is_morning"] = (
                    df["departure_hour"]
                    .between(6, 12)
                    .astype(np.int8)
                )

                df["is_afternoon"] = (
                    df["departure_hour"]
                    .between(12, 18)
                    .astype(np.int8)
                )

                df["is_night"] = (
                    (
                        df["departure_hour"] >= 18
                    ) |
                    (
                        df["departure_hour"] < 6
                    )
                ).astype(np.int8)

            # -----------------------------------------
            # SPEED ESTIMATION
            # -----------------------------------------

            if (
                "DISTANCE" in df.columns and
                "CRS_ELAPSED_TIME" in df.columns
            ):

                df["flight_speed_est"] = (

                    df["DISTANCE"].astype(np.float32)

                    /

                    (
                        df["CRS_ELAPSED_TIME"]
                        .astype(np.float32) + 1
                    )

                )


    # =====================================================
    # ROUTE FEATURES
    # =====================================================

    def add_route_features(self):

        df_train = self.data_loader.features_train

        df_test = self.data_loader.features_test

        if {"ORIGIN", "DEST"}.issubset(df_train.columns):

            df_train["route"] = (

                df_train["ORIGIN"].astype(str)

                + "_"

                + df_train["DEST"].astype(str)

            )

            df_test["route"] = (

                df_test["ORIGIN"].astype(str)

                + "_"

                + df_test["DEST"].astype(str)

            )

            route_freq = (
                df_train["route"]
                .value_counts()
            )

            df_train["route_frequency"] = (
                df_train["route"]
                .map(route_freq)
                .astype(np.float32)
            )

            df_test["route_frequency"] = (
                df_test["route"]
                .map(route_freq)
                .fillna(0)
                .astype(np.float32)
            )

        print("Route features created.")


    # =====================================================
    # INTERACTION FEATURES
    # =====================================================

    def add_interaction_features(self):

        df_train = self.data_loader.features_train

        df_test = self.data_loader.features_test

        features = [

            "DISTANCE",

            "departure_hour",

            "flight_speed_est"

        ]

        features = [

            f for f in features

            if f in df_train.columns

        ]

        # LIMITA FEATURES
        # evita explosão de memória
        max_interactions = 10

        interaction_count = 0

        for f1, f2 in combinations_with_replacement(
            features,
            2
        ):

            if f1 != f2:

                if interaction_count >= max_interactions:
                    break

                new_feature = f"{f1}_x_{f2}"

                df_train[new_feature] = (

                    df_train[f1]
                    .astype(np.float32)

                    *

                    df_train[f2]
                    .astype(np.float32)

                )

                df_test[new_feature] = (

                    df_test[f1]
                    .astype(np.float32)

                    *

                    df_test[f2]
                    .astype(np.float32)

                )

                interaction_count += 1

        print(
            f"{interaction_count} interaction "
            f"features created."
        )

In [8]:
#%% =====================================================
# PIPELINE EXECUTION
# =====================================================

data_loader = FlightDelayDataManipulator("../data/flights_sample_3m.csv")

print("\nBefore cleaning")
print(data_loader.features_train.shape)

cleaner = FlightDelayDataCleaning(data_loader)
cleaner.remove_duplicates()
cleaner.handle_missing_values()

print("\nAfter cleaning")
print(data_loader.features_train.shape)

preprocessing = FlightDelayPreprocessing(data_loader)

print("\nAfter preprocessing")
print(data_loader.features_train.shape)



LOADING DATASET
Original dataset shape: (3000000, 32)
After removing cancelled flights: (2920860, 32)

TOP CORRELATIONS WITH TARGET
FLIGHT_DELAYED             1.000000
ARR_DELAY                  0.590162
DEP_DELAY                  0.521520
TAXI_OUT                   0.253746
DEP_TIME                   0.178262
WHEELS_OFF                 0.176146
CRS_DEP_TIME               0.136524
TAXI_IN                    0.129988
CRS_ARR_TIME               0.111623
ELAPSED_TIME               0.089990
WHEELS_ON                  0.076617
DELAY_DUE_LATE_AIRCRAFT    0.075541
ARR_TIME                   0.066712
DELAY_DUE_CARRIER          0.051949
AIR_TIME                   0.047206
DELAY_DUE_NAS              0.039476
DISTANCE                   0.027477
CRS_ELAPSED_TIME           0.026899
FL_NUMBER                  0.026476
DIVERTED                   0.022775
Name: FLIGHT_DELAYED, dtype: float64

Removed 22 leakage columns.

TARGET DISTRIBUTION

Full Dataset:
FLIGHT_DELAYED
0    2405571
1     515289
Name

In [9]:
#%% =====================================================
# RUN EDA + VISUALIZATION
# =====================================================

eda = FlightDelayEDA(data_loader)
eda.run()



Generating correlation heatmap...
Correlation heatmap saved.
Flights per airline saved.
Delays per airline saved.
Delay percentage per airline saved.
ARR_DELAY not available → skipping
ARR_DELAY not available → skipping
Flights by airport saved.
ARR_DELAY not available → skipping
ARR_DELAY not available → skipping
Airport traffic saved.


In [ ]:
#%% 
# FEATURE ANALYSIS PIPELINE
# ------------------------------------------------

feature_analysis = FlightDelayFeatureAnalysisAndGenerator(data_loader)

# PCA
feature_analysis.perform_pca(
    explained_variance_threshold=0.80,
    plot_pca=True,
    add_pca=True
)

# novas features
feature_analysis.generate_features()

# importância
top_features = feature_analysis.relevant_feature_identification(
    num_features=10
)

# ------------------------------------------------
# CLEAN DATA BEFORE SAVING
# ------------------------------------------------

for dataset in [
    data_loader.features_train,
    data_loader.features_test
]:

    dataset.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    dataset = dataset.infer_objects(copy=False)

    dataset.fillna(0, inplace=True)

# ------------------------------------------------
# MEMORY OPTIMIZATION
# ------------------------------------------------

for dataset_name in [
    "features_train",
    "features_test"
]:

    dataset = getattr(data_loader, dataset_name)

    float_cols = dataset.select_dtypes(
        include=["float64"]
    ).columns

    dataset[float_cols] = dataset[
        float_cols
    ].astype(np.float32)

    setattr(data_loader, dataset_name, dataset)

# ------------------------------------------------
# SAVE DATASET
# ------------------------------------------------

with open("data_loader_with_new_features.pkl", "wb") as f:
    pickle.dump(
        data_loader,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

# ------------------------------------------------
# LOAD DATASET
# ------------------------------------------------

with open("data_loader_with_new_features.pkl", "rb") as f:
    data_loader_loaded = pickle.load(f)

# ------------------------------------------------
# INFO
# ------------------------------------------------

print("Training data shape:",
      data_loader_loaded.features_train.shape)

print("Training target shape:",
      data_loader_loaded.target_train.shape)

print("Testing data shape:",
      data_loader_loaded.features_test.shape)

print("Testing target shape:",
      data_loader_loaded.target_test.shape)

Selected 7 PCA components
PCA variance plot saved.
Route features created.
3 interaction features created.
Feature generation completed.


In [ ]:
#%% =====================================================
# SAVE DATASET
# =====================================================

os.makedirs("../output", exist_ok=True)

with open("../output/flight_delay_dataset.pkl", "wb") as f:

    pickle.dump(data_loader, f)

print("Dataset saved")


In [ ]:
#%% =====================================================
# LOAD DATASET
# =====================================================

with open("../output/flight_delay_dataset.pkl", "rb") as f:

    loaded_dataset = pickle.load(f)

print("Dataset loaded")
print(loaded_dataset.features_train.shape)

In [ ]:
#%% =====================================================
# KNN FROM SCRATCH
# =====================================================

class KNNFromScratch:

    def __init__(
        self,
        k=5,
        output_folder="../output/models"
    ):

        self.k = k

        self.X_train = None
        self.y_train = None

        self.output_folder = output_folder

        os.makedirs(
            self.output_folder,
            exist_ok=True
        )


    # =====================================================
    # TRAIN
    # =====================================================

    def fit(self, X, y):

        print("\n==============================")
        print("TRAINING KNN FROM SCRATCH")
        print("==============================")

        self.X_train = np.array(X)

        self.y_train = np.array(y)

        print(f"Training samples : {len(self.X_train)}")
        print(f"Number of features: {self.X_train.shape[1]}")
        print(f"k value           : {self.k}")

        print("\nkNN from scratch trained.")


    # =====================================================
    # EUCLIDEAN DISTANCE
    # =====================================================

    def euclidean_distance(self, x1, x2):

        return np.sqrt(
            np.sum((x1 - x2) ** 2)
        )


    # =====================================================
    # PREDICT SINGLE SAMPLE
    # =====================================================

    def predict_single(self, x):

        distances = []

        for train_sample in self.X_train:

            distance = self.euclidean_distance(
                x,
                train_sample
            )

            distances.append(distance)

        distances = np.array(distances)

        k_indices = distances.argsort()[:self.k]

        k_labels = self.y_train[k_indices]

        prediction = np.bincount(k_labels).argmax()

        return prediction


    # =====================================================
    # PREDICT
    # =====================================================

    def predict(self, X):

        print("\nStarting predictions...")

        X = np.array(X)

        predictions = []

        total_samples = len(X)

        for index, x in enumerate(X):

            prediction = self.predict_single(x)

            predictions.append(prediction)

            if (index + 1) % 100 == 0:

                print(
                    f"Processed "
                    f"{index + 1}/{total_samples} samples"
                )

        print("Predictions completed.")

        return np.array(predictions)


    # =====================================================
    # EVALUATE
    # =====================================================

    def evaluate(self, X_test, y_test):

        print("\n==============================")
        print("EVALUATING KNN")
        print("==============================")

        predictions = self.predict(X_test)

        # ------------------------------------------------
        # METRICS
        # ------------------------------------------------

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        balanced_accuracy = balanced_accuracy_score(
            y_test,
            predictions
        )

        precision = precision_score(
            y_test,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            predictions,
            zero_division=0
        )

        # ------------------------------------------------
        # RESULTS
        # ------------------------------------------------

        print(f"\nAccuracy          : {accuracy:.4f}")
        print(f"Balanced Accuracy : {balanced_accuracy:.4f}")
        print(f"Precision         : {precision:.4f}")
        print(f"Recall            : {recall:.4f}")
        print(f"F1-Score          : {f1:.4f}")

        # ------------------------------------------------
        # PREDICTION DISTRIBUTION
        # ------------------------------------------------

        unique, counts = np.unique(
            predictions,
            return_counts=True
        )

        distribution = dict(
            zip(unique, counts)
        )

        print("\nPrediction Distribution:")
        print(distribution)

        # ------------------------------------------------
        # REPORT
        # ------------------------------------------------

        print("\nClassification Report:\n")

        print(
            classification_report(
                y_test,
                predictions,
                zero_division=0
            )
        )

        # ------------------------------------------------
        # CONFUSION MATRIX
        # ------------------------------------------------

        cm = confusion_matrix(
            y_test,
            predictions,
            normalize="true"
        )

        plt.figure(figsize=(6,5))

        sns.heatmap(
            cm,
            annot=True,
            fmt=".2f",
            cmap="Blues"
        )

        plt.title("kNN From Scratch")

        plt.xlabel("Predicted")
        plt.ylabel("Actual")

        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/"
            f"knn_from_scratch_confusion_matrix.png"
        )

        plt.close()

        print("Confusion matrix saved.")

        return {

            "model": "kNN From Scratch",

            "accuracy": accuracy,

            "balanced_accuracy": balanced_accuracy,

            "precision": precision,

            "recall": recall,

            "f1_score": f1,

            "roc_auc": 0,

            "probabilities": None
        }

    def predict_proba(self, X):
        probs = []

        for x in X:
            neighbors = self._get_neighbors(x)

            labels = [n[-1] for n in neighbors]

            prob_class_1 = sum(labels) / len(labels)

            probs.append(prob_class_1)

        return np.array(probs)
    # =====================================================
    # SAVE MODEL
    # =====================================================

    def save_model(self):

        print("\nSaving model...")

        model_path = (
            f"{self.output_folder}/"
            f"knn_from_scratch.pkl"
        )

        with open(model_path, "wb") as f:

            pickle.dump(
                self,
                f
            )

        print(f"Model saved at {model_path}")


In [ ]:
# =====================================================
# PREPARE DATA
# =====================================================

X_train = data_loader.features_train.select_dtypes(
    include=np.number
)

X_test = data_loader.features_test.select_dtypes(
    include=np.number
)

y_train = data_loader.target_train
y_test = data_loader.target_test

# =====================================================
# CLEAN DATA
# =====================================================

X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

# =====================================================
# SCALE DATA
# =====================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

# =====================================================
# USE SAMPLE (IMPORTANT FOR PERFORMANCE)
# =====================================================

sample_size = 10000

X_train = X_train[:sample_size]

y_train = y_train.iloc[:sample_size]

# IMPORTANT FIX FOR PERFORMANCE

X_test = X_test[:20000]

y_test = y_test.iloc[:20000]

# =====================================================
# TRAIN MODEL
# =====================================================
results = []

knn_scratch = KNNFromScratch(k=5)


knn_scratch.fit(
    X_train,
    y_train
)

# =====================================================
# EVALUATE
# =====================================================

# # IMPORTANT: probabilities for ROC
if hasattr(knn_scratch, "predict_proba"):
    probabilities = knn_scratch.predict_proba(X_test)
else:
    probabilities = None

# ================================
# METRICS
# ================================
knn_metrics = knn_scratch.evaluate(X_test, y_test)

# ================================
# SAVE IN UNIFIED FORMAT
# ================================
results.append({
    "model": "kNN From Scratch",
    "accuracy": knn_metrics["accuracy"],
    "precision": knn_metrics["precision"],
    "recall": knn_metrics["recall"],
    "f1_score": knn_metrics["f1_score"],
    "roc_auc": knn_metrics.get("roc_auc"),
    "probabilities": probabilities
})

In [ ]:
#%% =====================================================
# BASE MODEL
# =====================================================

class BaseModel:

    def __init__(
        self,
        data_loader,
        model_name="model",
        scale_data=False,
        output_folder="../output/models"
    ):

        self.data_loader = data_loader

        self.model_name = model_name
        self.scale_data = scale_data

        self.output_folder = output_folder

        os.makedirs(
            self.output_folder,
            exist_ok=True
        )

        # ------------------------------------------------
        # DATA
        # ------------------------------------------------

        self.X_train = (
            self.data_loader.features_train.copy()
        )

        self.X_test = (
            self.data_loader.features_test.copy()
        )

        self.y_train = (
            self.data_loader.target_train.copy()
        )

        self.y_test = (
            self.data_loader.target_test.copy()
        )

        # ------------------------------------------------
        # NUMERIC ONLY
        # ------------------------------------------------

        self.X_train = self.X_train.select_dtypes(
            include=np.number
        )

        self.X_test = self.X_test.select_dtypes(
            include=np.number
        )

        # ------------------------------------------------
        # CLEAN DATA
        # ------------------------------------------------

        self.X_train.replace(
            [np.inf, -np.inf],
            np.nan,
            inplace=True
        )

        self.X_test.replace(
            [np.inf, -np.inf],
            np.nan,
            inplace=True
        )

        self.X_train.fillna(0, inplace=True)
        self.X_test.fillna(0, inplace=True)

        # ------------------------------------------------
        # ALIGN FEATURES
        # ------------------------------------------------

        self.X_train, self.X_test = self.X_train.align(
            self.X_test,
            join="left",
            axis=1,
            fill_value=0
        )

        # ------------------------------------------------
        # SCALING
        # ------------------------------------------------

        self.scaler = None

        if self.scale_data:

            self.scaler = StandardScaler()

            self.X_train = self.scaler.fit_transform(
                self.X_train
            )

            self.X_test = self.scaler.transform(
                self.X_test
            )

        self.model = None


    # =====================================================
    # TRAIN
    # =====================================================

    def train(self):

        print(f"\nTraining {self.model_name}...")

        self.model.fit(
            self.X_train,
            self.y_train
        )

        print(f"{self.model_name} trained.")


    # =====================================================
    # PREDICT
    # =====================================================

    def predict(self):

        return self.model.predict(self.X_test)


    # =====================================================
    # EVALUATE
    # =====================================================

    def evaluate(self):


        predictions = self.predict()
        # ------------------------------------------------
        # METRICS
        # ------------------------------------------------

        accuracy = accuracy_score(
            self.y_test,
            predictions
        )

        balanced_accuracy = balanced_accuracy_score(
            self.y_test,
            predictions
        )

        precision = precision_score(
            self.y_test,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            self.y_test,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            self.y_test,
            predictions,
            zero_division=0
        )

        # ------------------------------------------------
        # ROC AUC
        # ------------------------------------------------

        if hasattr(self.model, "predict_proba"):

            probabilities = self.model.predict_proba(
                self.X_test
            )[:, 1]

            roc_auc = roc_auc_score(
                self.y_test,
                probabilities
            )

        else:

            probabilities = None
            roc_auc = 0

        # ------------------------------------------------
        # RESULTS
        # ------------------------------------------------

        print(f"\nAccuracy          : {accuracy:.4f}")
        print(f"Balanced Accuracy : {balanced_accuracy:.4f}")
        print(f"Precision         : {precision:.4f}")
        print(f"Recall            : {recall:.4f}")
        print(f"F1-Score          : {f1:.4f}")
        print(f"ROC-AUC           : {roc_auc:.4f}")

        # ------------------------------------------------
        # PREDICTION DISTRIBUTION
        # ------------------------------------------------

        unique, counts = np.unique(
            predictions,
            return_counts=True
        )

        distribution = dict(
            zip(unique, counts)
        )

        print("\nPrediction Distribution:")
        print(distribution)

        # ------------------------------------------------
        # CLASSIFICATION REPORT
        # ------------------------------------------------

        print("\nClassification Report:\n")

        print(
            classification_report(
                self.y_test,
                predictions,
                zero_division=0
            )
        )

        # ------------------------------------------------
        # CONFUSION MATRIX
        # ------------------------------------------------

        self.plot_confusion_matrix(predictions)

        return {

            "model": self.model_name,

            "accuracy": accuracy,

            "balanced_accuracy": balanced_accuracy,

            "precision": precision,

            "recall": recall,

            "f1_score": f1,

            "roc_auc": roc_auc,

            "probabilities": probabilities
            }

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================

    def plot_confusion_matrix(self, predictions):

        cm = confusion_matrix(
            self.y_test,
            predictions,
            normalize="true"
        )

        plt.figure(figsize=(6,5))

        sns.heatmap(
            cm,
            annot=True,
            fmt=".2f",
            cmap="Blues"
        )

        plt.title(
            f"Confusion Matrix - {self.model_name}"
        )

        plt.xlabel("Predicted")
        plt.ylabel("Actual")

        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/"
            f"{self.model_name}_confusion_matrix.png"
        )

        plt.close()

        print("Confusion matrix saved.")


    # =====================================================
    # FEATURE IMPORTANCE
    # =====================================================

    def plot_feature_importance(self, top_n=15):

        if not hasattr(
            self.model,
            "feature_importances_"
        ):

            print(
                f"{self.model_name} "
                f"does not support feature importance."
            )

            return

        feature_names = (
            self.data_loader.features_train
            .select_dtypes(include=np.number)
            .columns
        )

        importance_df = pd.DataFrame({

            "feature": feature_names,

            "importance":
                self.model.feature_importances_

        })

        importance_df = (
            importance_df
            .sort_values(
                by="importance",
                ascending=False
            )
            .head(top_n)
        )

        plt.figure(figsize=(10,6))

        sns.barplot(
            data=importance_df,
            x="importance",
            y="feature"
        )

        plt.title(
            f"Feature Importance - {self.model_name}"
        )

        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/"
            f"{self.model_name}_feature_importance.png"
        )

        plt.close()

        print("Feature importance saved.")


    # =====================================================
    # SAVE MODEL
    # =====================================================

    def save_model(self):

        model_path = (
            f"{self.output_folder}/"
            f"{self.model_name}.pkl"
        )

        with open(model_path, "wb") as f:

            pickle.dump(
                self.model,
                f
            )

        print(f"Model saved at {model_path}")

In [ ]:
class RandomForestModel(BaseModel):

    def __init__(
        self,
        data_loader,
        n_estimators=100,
        max_depth=None
    ):

        super().__init__(
            data_loader=data_loader,
            model_name="random_forest",
            scale_data=False
        )

        self.model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
            n_jobs=-1
        )

In [ ]:
models = [

    RandomForestModel(
        data_loader,
        n_estimators=200,
        max_depth=15
    )
]

results = []

for model in models:

    model.train()

    metrics = model.evaluate()

    model.save_model()

    if hasattr(model.model, "feature_importances_"):

        model.plot_feature_importance()

    # ------------------------------------------------
    # ROC PROBABILITIES
    # ------------------------------------------------

    probabilities = (
        model.model.predict_proba(
            model.X_test
        )[:, 1]
    )

    # ------------------------------------------------
    # SAVE RESULTS
    # ------------------------------------------------

    results.append({

        "model": model.model_name,

        "accuracy": metrics["accuracy"],

        "precision": metrics["precision"],

        "recall": metrics["recall"],

        "f1_score": metrics["f1_score"],

        "roc_auc": roc_auc_score(
            model.y_test,
            probabilities
        ),

        "probabilities": probabilities
    })

In [ ]:
class FlightDelayDeepLearningModel:

    def __init__(
        self,
        data_loader,
        output_folder="../output/deep_learning"
    ):

        self.data_loader = data_loader
        self.output_folder = output_folder

        os.makedirs(self.output_folder, exist_ok=True)

        # ------------------------------------------------
        # DATA
        # ------------------------------------------------
        self.X_train = (
            self.data_loader.features_train
            .select_dtypes(include=np.number)
            .copy()
        )

        self.X_test = (
            self.data_loader.features_test
            .select_dtypes(include=np.number)
            .copy()
        )

        self.y_train = self.data_loader.target_train.copy()
        self.y_test = self.data_loader.target_test.copy()

        # ------------------------------------------------
        # REMOVE LEAKAGE
        # ------------------------------------------------
        leakage_columns = [
            "ARR_DELAY"
        ]

        self.X_train = self.X_train.drop(
            columns=leakage_columns,
            errors="ignore"
        )

        self.X_test = self.X_test.drop(
            columns=leakage_columns,
            errors="ignore"
        )

        # ------------------------------------------------
        # CLEAN DATA
        # ------------------------------------------------
        self.X_train.replace(
            [np.inf, -np.inf],
            np.nan,
            inplace=True
        )

        self.X_test.replace(
            [np.inf, -np.inf],
            np.nan,
            inplace=True
        )

        self.X_train.fillna(0, inplace=True)
        self.X_test.fillna(0, inplace=True)

        # ------------------------------------------------
        # SCALE
        # ------------------------------------------------
        self.scaler = StandardScaler()

        self.X_train = self.scaler.fit_transform(
            self.X_train
        )

        self.X_test = self.scaler.transform(
            self.X_test
        )

        self.model = None


    # =====================================================
    # BUILD MODEL
    # =====================================================

    def build_model(self):

        self.model = Sequential([

            Dense(
                128,
                activation="relu",
                input_shape=(self.X_train.shape[1],)
            ),

            Dropout(0.3),

            Dense(
                64,
                activation="relu"
            ),

            Dropout(0.3),

            Dense(
                32,
                activation="relu"
            ),

            Dense(
                1,
                activation="sigmoid"
            )

        ])

        self.model.compile(
            optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

        print("Deep Learning model created.")


    # =====================================================
    # TRAIN
    # =====================================================

    def train(
        self,
        epochs=20,
        batch_size=256
    ):

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )

        history = self.model.fit(

            self.X_train,
            self.y_train,

            validation_split=0.2,

            epochs=epochs,
            batch_size=batch_size,

            callbacks=[early_stopping],

            verbose=1
        )

        return history
    def predict_proba(self, X):
        return self.model.predict(X)  # sigmoid / softmax já tratado

    # =====================================================
    # EVALUATE
    # =====================================================

    def evaluate(self):

        print("\n==============================")
        print("EVALUATING DEEP LEARNING")
        print("==============================")

        # ------------------------------------------------
        # PROBABILITIES
        # ------------------------------------------------

        probabilities = self.model.predict(
            self.X_test,
            verbose=0
        ).flatten()

        # ------------------------------------------------
        # THRESHOLD
        # ------------------------------------------------

        predictions = (
            probabilities > 0.35
        ).astype(int)

        # ------------------------------------------------
        # METRICS
        # ------------------------------------------------

        accuracy = accuracy_score(
            self.y_test,
            predictions
        )

        balanced_accuracy = balanced_accuracy_score(
            self.y_test,
            predictions
        )

        precision = precision_score(
            self.y_test,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            self.y_test,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            self.y_test,
            predictions,
            zero_division=0
        )

        roc_auc = roc_auc_score(
            self.y_test,
            probabilities
        )

        # ------------------------------------------------
        # RESULTS
        # ------------------------------------------------

        print(f"\nAccuracy          : {accuracy:.4f}")
        print(f"Balanced Accuracy : {balanced_accuracy:.4f}")
        print(f"Precision         : {precision:.4f}")
        print(f"Recall            : {recall:.4f}")
        print(f"F1-Score          : {f1:.4f}")
        print(f"ROC-AUC           : {roc_auc:.4f}")

        # ------------------------------------------------
        # PREDICTION DISTRIBUTION
        # ------------------------------------------------

        unique, counts = np.unique(
            predictions,
            return_counts=True
        )

        distribution = dict(
            zip(unique, counts)
        )

        print("\nPrediction Distribution:")
        print(distribution)

        # ------------------------------------------------
        # REPORT
        # ------------------------------------------------

        print("\nClassification Report:\n")

        print(
            classification_report(
                self.y_test,
                predictions,
                zero_division=0
            )
        )

        # ------------------------------------------------
        # CONFUSION MATRIX
        # ------------------------------------------------

        cm = confusion_matrix(
            self.y_test,
            predictions,
            normalize="true"
        )

        plt.figure(figsize=(6,5))

        sns.heatmap(
            cm,
            annot=True,
            fmt=".2f",
            cmap="Blues"
        )

        plt.title("Deep Learning Confusion Matrix")

        plt.xlabel("Predicted")
        plt.ylabel("Actual")

        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/"
            f"deep_learning_confusion_matrix.png"
        )

        plt.close()

        print("Confusion matrix saved.")

        return {

            "model": "Deep Learning",

            "accuracy": accuracy,

            "balanced_accuracy": balanced_accuracy,

            "precision": precision,

            "recall": recall,

            "f1_score": f1,

            "roc_auc": roc_auc,

            "probabilities": probabilities
        }


    # =====================================================
    # SAVE MODEL
    # =====================================================

    def save_model(self):

        self.model.save(
            f"{self.output_folder}/deep_learning_model.h5"
        )

        print("Deep Learning model saved.")

In [ ]:
#%% =====================================================
# RUN DEEP LEARNING
# =====================================================

deep_learning_model = FlightDelayDeepLearningModel(
    data_loader
)

deep_learning_model.build_model()

history = deep_learning_model.train(
    epochs=20,
    batch_size=256
)
y_pred = deep_learning_model.predict(X_test)
y_prob = deep_learning_model.predict_proba(X_test)

results_dl = deep_learning_model.evaluate()

results.append({
    "model": "Deep Learning",
    "accuracy": results_dl["accuracy"],
    "precision": results_dl["precision"],
    "recall": results_dl["recall"],
    "f1_score": results_dl["f1_score"],
    "roc_auc": results_dl.get("roc_auc"),
    "probabilities": y_prob
})

deep_learning_model.save_model()



In [ ]:
#%% =====================================================
# CLUSTERING ANALYSIS
# =====================================================

class FlightDelayClustering:

    """
    Clustering analysis for flight delay patterns.

    Algorithm:
    - KMeans

    Features:
    - PCA visualization
    - Elbow method
    - Silhouette Score
    - Davies-Bouldin Score
    - Cluster summary
    """

    def __init__(
        self,
        data_loader,
        output_folder="../output/clustering"
    ):

        self.data_loader = data_loader

        self.output_folder = output_folder

        os.makedirs(
            self.output_folder,
            exist_ok=True
        )

        self.scaler = StandardScaler()

        self.X = None
        self.X_scaled = None
        self.X_pca = None

        self.cluster_labels = None

        self.feature_names = None

        self.model = None


    # =====================================================
    # PREPARE DATA
    # =====================================================

    def prepare_data(self):

        print("\n==============================")
        print("PREPARING CLUSTERING DATA")
        print("==============================")

        df = self.data_loader.features_train.copy()

        # ------------------------------------------------
        # ONLY NUMERIC FEATURES
        # ------------------------------------------------

        df = df.select_dtypes(
            include=np.number
        )

        print(f"Initial shape: {df.shape}")

        # ------------------------------------------------
        # REMOVE LEAKAGE FEATURES
        # ------------------------------------------------

        leakage_columns = [

            "ARR_DELAY",
            "DEP_DELAY",
            "AIRLINE_CODE",
            "ARR_TIME",
            "DEP_TIME",

            "WHEELS_ON",
            "WHEELS_OFF",

            "TAXI_IN",
            "TAXI_OUT",

            "AIR_TIME",
            "ELAPSED_TIME",

            "DELAY_DUE_CARRIER",
            "DELAY_DUE_WEATHER",
            "DELAY_DUE_NAS",
            "DELAY_DUE_SECURITY",
            "DELAY_DUE_LATE_AIRCRAFT",

            "CANCELLED",
            "DIVERTED"
        ]

        leakage_columns = [

            col for col in leakage_columns
            if col in df.columns
        ]

        df.drop(
            columns=leakage_columns,
            inplace=True
        )

        print(
            f"Removed {len(leakage_columns)} leakage columns."
        )

        # ------------------------------------------------
        # CLEAN DATA
        # ------------------------------------------------

        df.replace(
            [np.inf, -np.inf],
            np.nan,
            inplace=True
        )

        df.dropna(
            axis=1,
            how="all",
            inplace=True
        )

        df.fillna(0, inplace=True)

        # ------------------------------------------------
        # REMOVE CONSTANT COLUMNS
        # ------------------------------------------------

        constant_cols = [

            col for col in df.columns
            if df[col].nunique() <= 1
        ]

        if constant_cols:

            df.drop(
                columns=constant_cols,
                inplace=True
            )

            print(
                f"Removed {len(constant_cols)} constant columns."
            )

        # ------------------------------------------------
        # MEMORY OPTIMIZATION
        # ------------------------------------------------

        float_cols = df.select_dtypes(
            include=["float64"]
        ).columns

        df[float_cols] = df[
            float_cols
        ].astype(np.float32)

        # ------------------------------------------------
        # SAMPLE DATA
        # ------------------------------------------------

        sample_size = min(50000, len(df))

        df = df.sample(
            n=sample_size,
            random_state=42
        )

        print(f"Sample size used: {sample_size}")

        # ------------------------------------------------
        # FINAL CHECKS
        # ------------------------------------------------

        if df.empty:

            raise ValueError(
                "Dataset became empty after cleaning."
            )

        if df.shape[1] < 2:

            raise ValueError(
                "Not enough features for clustering."
            )

        self.feature_names = df.columns

        self.X = df

        # ------------------------------------------------
        # SCALE DATA
        # ------------------------------------------------

        self.X_scaled = self.scaler.fit_transform(
            df
        )

        self.X_scaled = np.nan_to_num(
            self.X_scaled
        )

        print("Data prepared successfully.")
        print(f"Final shape: {self.X_scaled.shape}")


    # =====================================================
    # PCA REDUCTION
    # =====================================================

    def apply_pca(self, n_components=2):

        print("\nApplying PCA...")

        pca = PCA(
            n_components=n_components,
            random_state=42
        )

        self.X_pca = pca.fit_transform(
            self.X_scaled
        )

        explained = np.sum(
            pca.explained_variance_ratio_
        )

        print(
            f"PCA completed "
            f"(Explained variance: {explained:.4f})"
        )


    # =====================================================
    # ELBOW METHOD
    # =====================================================

    def plot_elbow_method(self, max_k=10):

        print("\nGenerating Elbow Method...")

        inertias = []

        k_values = range(2, max_k + 1)

        for k in k_values:

            model = KMeans(
                n_clusters=k,
                random_state=42,
                n_init=10
            )

            model.fit(self.X_scaled)

            inertias.append(
                model.inertia_
            )

            print(f"k={k} completed.")

        plt.figure(figsize=(8,5))

        plt.plot(
            k_values,
            inertias,
            marker="o"
        )

        plt.title("Elbow Method")

        plt.xlabel("Number of Clusters")
        plt.ylabel("Inertia")

        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/elbow_method.png"
        )

        plt.close()

        print("Elbow method saved.")


    # =====================================================
    # TRAIN KMEANS
    # =====================================================

    def train_kmeans(self, n_clusters=3):

        print(
            f"\nTraining KMeans "
            f"with {n_clusters} clusters..."
        )

        self.model = KMeans(

            n_clusters=n_clusters,

            random_state=42,

            n_init=10
        )

        self.cluster_labels = self.model.fit_predict(
            self.X_scaled
        )

        print("KMeans trained successfully.")


    # =====================================================
    # EVALUATE KMEANS
    # =====================================================

    def evaluate_kmeans(self):

        silhouette = silhouette_score(
            self.X_scaled,
            self.cluster_labels
        )

        db_score = davies_bouldin_score(
            self.X_scaled,
            self.cluster_labels
        )

        inertia = self.model.inertia_

        print("\n==============================")
        print("KMEANS EVALUATION")
        print("==============================")

        print(
            f"Silhouette Score     : "
            f"{silhouette:.4f}"
        )

        print(
            f"Davies-Bouldin Score : "
            f"{db_score:.4f}"
        )

        print(
            f"Inertia              : "
            f"{inertia:.4f}"
        )

        return {

            "model": "KMeans",

            "silhouette_score": silhouette,

            "davies_bouldin_score": db_score,

            "inertia": inertia
        }


    # =====================================================
    # PLOT CLUSTERS
    # =====================================================

    def plot_clusters(self, title="KMeans_Clusters"):

        if self.X_pca is None:

            raise ValueError(
                "Run apply_pca() first."
            )

        plt.figure(figsize=(10,7))

        sns.scatterplot(

            x=self.X_pca[:, 0],

            y=self.X_pca[:, 1],

            hue=self.cluster_labels,

            palette="tab10",

            s=40
        )

        plt.title(title)

        plt.xlabel("PCA Component 1")
        plt.ylabel("PCA Component 2")

        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/{title}.png"
        )

        plt.close()

        print(f"{title} plot saved.")


    # =====================================================
    # CLUSTER DISTRIBUTION
    # =====================================================

    def plot_cluster_distribution(
        self,
        n_clusters
    ):

        plt.figure(figsize=(7,5))

        sns.countplot(
            x=self.cluster_labels
        )

        plt.title(
            f"Cluster Distribution ({n_clusters})"
        )

        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/"
            f"cluster_distribution_{n_clusters}.png"
        )

        plt.close()


    # =====================================================
    # CLUSTER SUMMARY
    # =====================================================

    def cluster_summary(self, n_clusters):

        print("\nGenerating cluster summary...")

        df = pd.DataFrame(
            self.X_scaled,
            columns=self.feature_names
        )

        df["cluster"] = self.cluster_labels

        summary = df.groupby(
            "cluster"
        ).mean()

        summary.to_csv(
            f"{self.output_folder}/"
            f"kmeans_{n_clusters}_summary.csv"
        )

        print("Cluster summary saved.")

        return summary


   

In [ ]:
# ------------------------------------------------
# CREATE OUTPUT FOLDER
# ------------------------------------------------

os.makedirs(
    "../output/model_comparison",
    exist_ok=True
)

# ------------------------------------------------
# INITIALIZE CLUSTERING
# ------------------------------------------------

clustering = FlightDelayClustering(
    data_loader
)

# ------------------------------------------------
# PREPARE DATA
# ------------------------------------------------

clustering.prepare_data()

# ------------------------------------------------
# PCA
# ------------------------------------------------

clustering.apply_pca()

# ------------------------------------------------
# ELBOW METHOD
# ------------------------------------------------

clustering.plot_elbow_method()

# ------------------------------------------------
# TEST DIFFERENT K VALUES
# ------------------------------------------------

clustering_results = []

for k in [2, 3, 4, 5, 6]:

    print("\n==============================")
    print(f"KMEANS WITH {k} CLUSTERS")
    print("==============================")

    # ----------------------------------------
    # TRAIN KMEANS
    # ----------------------------------------

    clustering.train_kmeans(
        n_clusters=k
    )

    # ----------------------------------------
    # EVALUATE
    # ----------------------------------------

    results = clustering.evaluate_kmeans()

    # ----------------------------------------
    # VISUALIZATION
    # ----------------------------------------

    clustering.plot_cluster_distribution(
    n_clusters=k
    )

    # ----------------------------------------
    # CLUSTER SUMMARY
    # ----------------------------------------

    clustering.cluster_summary(
        n_clusters=k
    )

    # ----------------------------------------
    # SAVE METRICS
    # ----------------------------------------

    clustering_results.append({

        "model": "KMeans",

        "n_clusters": k,

        "silhouette_score":
            results["silhouette_score"],

        "davies_bouldin_score":
            results["davies_bouldin_score"],

        "inertia":
            results["inertia"]

    })

# ------------------------------------------------
# CREATE RESULTS DATAFRAME
# ------------------------------------------------

clustering_df = pd.DataFrame(
    clustering_results
)

# ------------------------------------------------
# SORT BY SILHOUETTE
# ------------------------------------------------

clustering_df = clustering_df.sort_values(
    by="silhouette_score",
    ascending=False
)

# ------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------

print("\n==============================")
print("KMEANS RESULTS")
print("==============================")

print(clustering_df)

# ------------------------------------------------
# SAVE RESULTS CSV
# ------------------------------------------------

clustering_df.to_csv(
    "../output/model_comparison/kmeans_results.csv",
    index=False
)

print("\nResults saved.")

# ------------------------------------------------
# SILHOUETTE COMPARISON
# ------------------------------------------------

plt.figure(figsize=(8,5))

sns.barplot(
    data=clustering_df,
    x="n_clusters",
    y="silhouette_score"
)

plt.title(
    "KMeans Silhouette Score Comparison"
)

plt.xlabel(
    "Number of Clusters"
)

plt.ylabel(
    "Silhouette Score"
)

plt.tight_layout()

plt.savefig(
    "../output/model_comparison/kmeans_silhouette_comparison.png"
)

plt.close()

# ------------------------------------------------
# DAVIES BOULDIN COMPARISON
# ------------------------------------------------

plt.figure(figsize=(8,5))

sns.barplot(
    data=clustering_df,
    x="n_clusters",
    y="davies_bouldin_score"
)

plt.title(
    "KMeans Davies-Bouldin Comparison"
)

plt.xlabel(
    "Number of Clusters"
)

plt.ylabel(
    "Davies-Bouldin Score"
)

plt.tight_layout()

plt.savefig(
    "../output/model_comparison/kmeans_davies_bouldin_comparison.png"
)

plt.close()

print("\nAll clustering outputs saved successfully.")

In [ ]:
class FlightDelayModelComparison:
    """
    Compare multiple machine learning models.

    Features:
    - Stores model results (flexible dict-based)
    - Creates comparison tables
    - Plots performance metrics
    - Plots ROC curves (when available)
    - Generates insights
    """

    def __init__(self, output_folder="../output/model_comparison"):

        self.output_folder = output_folder
        os.makedirs(self.output_folder, exist_ok=True)

        self.results = []

    # =====================================================
    # ADD MODEL RESULT (GENERIC VERSION)
    # =====================================================

    def add_model_result(self, model_name, metrics: dict):

        entry = {
            "Model": model_name,
            **metrics
        }

        self.results.append(entry)

        print(f"{model_name} results added.")

    # =====================================================
    # CREATE COMPARISON TABLE
    # =====================================================

    def create_comparison_table(self):

        if not self.results:
            raise ValueError("No results to compare.")

        results_df = pd.DataFrame(self.results)

        # Sort by F1-Score if exists, otherwise Accuracy
        sort_col = "F1-Score" if "F1-Score" in results_df.columns else "Accuracy"

        results_df = results_df.sort_values(by=sort_col, ascending=False)

        print("\n==============================")
        print("MODEL COMPARISON")
        print("==============================")
        print(results_df)

        results_df.to_csv(
            f"{self.output_folder}/model_comparison.csv",
            index=False
        )

        return results_df

    # =====================================================
    # PLOT METRICS COMPARISON
    # =====================================================

    def plot_model_performance(self):

        results_df = pd.DataFrame(self.results)

        metrics = [
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC"
        ]

        for metric in metrics:

            if metric not in results_df.columns:
                continue

            plt.figure(figsize=(10, 6))

            sns.barplot(
                data=results_df,
                x="Model",
                y=metric
            )

            plt.title(f"{metric} Comparison")
            plt.xticks(rotation=45)
            plt.tight_layout()

            plt.savefig(
                f"{self.output_folder}/{metric.lower()}_comparison.png"
            )

            plt.close()

        print("Performance comparison plots saved.")

    # =====================================================
    # ROC CURVE (SAFE VERSION)
    # =====================================================

    def plot_roc_curve(self, y_test, model_predictions):

        """
        model_predictions:
        [
            ("Model Name", probabilities),
        ]
        """

        if not model_predictions:
            print("No probability-based models for ROC curve.")
            return

        plt.figure(figsize=(8, 6))

        for model_name, probabilities in model_predictions:

            fpr, tpr, _ = roc_curve(y_test, probabilities)
            auc_score = roc_auc_score(y_test, probabilities)

            plt.plot(
                fpr,
                tpr,
                label=f"{model_name} (AUC={auc_score:.3f})"
            )

        plt.plot([0, 1], [0, 1], linestyle="--")

        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve Comparison")
        plt.legend()

        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/roc_curve_comparison.png"
        )

        plt.close()

        print("ROC curve comparison saved.")

    # =====================================================
    # CLUSTERING COMPARISON
    # =====================================================

    def plot_clustering_comparison(self, clustering_df):

        plt.figure(figsize=(8, 5))

        sns.barplot(
            data=clustering_df,
            x="Model",
            y="Silhouette Score"
        )

        plt.title("Clustering Comparison")
        plt.tight_layout()

        plt.savefig(
            f"{self.output_folder}/clustering_comparison.png"
        )

        plt.close()

        print("Clustering comparison saved.")

    # =====================================================
    # INSIGHTS
    # =====================================================

    def summarize_results(self):

        if not self.results:
            print("No results to summarize.")
            return

        df = pd.DataFrame(self.results)

        print("\n==============================")
        print("PROJECT INSIGHTS")
        print("==============================")

        # Best F1 (if exists)
        if "F1-Score" in df.columns:
            best_f1 = df.sort_values("F1-Score", ascending=False).iloc[0]
            print(f"\nBest F1-Score Model: {best_f1['Model']} ({best_f1['F1-Score']:.4f})")

        # Best Accuracy
        if "Accuracy" in df.columns:
            best_acc = df.sort_values("Accuracy", ascending=False).iloc[0]
            print(f"Best Accuracy Model: {best_acc['Model']} ({best_acc['Accuracy']:.4f})")

        # Best ROC-AUC
        if "ROC-AUC" in df.columns:
            best_auc = df.sort_values("ROC-AUC", ascending=False).iloc[0]
            print(f"Best ROC-AUC Model: {best_auc['Model']} ({best_auc['ROC-AUC']:.4f})")

In [ ]:
#%% =====================================================
# MODEL COMPARISON PIPELINE
# =====================================================

comparison = FlightDelayModelComparison()

# ------------------------------------------------
# ADD ALL MODEL RESULTS
# ------------------------------------------------

comparison.add_model_result(
    model.model_name,
    {
        "Accuracy": metrics["accuracy"],
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1-Score": metrics["f1_score"],
        "ROC-AUC": roc_auc_score(model.y_test, probabilities)
    }
)
comparison.add_model_result(
    "Deep Learning",
    {
        "Accuracy": results_dl["accuracy"],
        "Precision": results_dl["precision"],
        "Recall": results_dl["recall"],
        "F1-Score": results_dl["f1_score"],
        "ROC-AUC": results_dl.get("roc_auc")
    }
)
comparison.add_model_result(
    "kNN From Scratch",
    {
        "Accuracy": knn_metrics["accuracy"],
        "Precision": knn_metrics["precision"],
        "Recall": knn_metrics["recall"],
        "F1-Score": knn_metrics["f1_score"],
        "ROC-AUC": knn_metrics.get("roc_auc")
    }
)
# ------------------------------------------------
# CREATE COMPARISON TABLE
# ------------------------------------------------

comparison_table = (
    comparison.create_comparison_table()
)

# ------------------------------------------------
# PERFORMANCE PLOTS
# ------------------------------------------------

comparison.plot_model_performance()

# ------------------------------------------------
# ROC CURVE
# ------------------------------------------------

roc_models = []

for result in results:

    if (
        "probabilities" in result
        and result["probabilities"] is not None
    ):

        roc_models.append(

            (
                result["model"],
                result["probabilities"]
            )
        )

comparison.plot_roc_curve(

    y_test=data_loader.target_test,

    model_predictions=roc_models
)



comparison.summarize_results()